In [ ]:
# This file is used to build utilities for the baseline model

from transformers import DistilBertTokenizerFast
from datasets import load_dataset
import evaluate
class BaseUtilsImdb:
    '''
    This class is used to build utilities for the baseline model
    The split defines the percentage of the data to be used for validation from the training data
    '''
    def __init__(self,split=0.1,random_seed=42,clean_text=False,prompt_tokens=0):
        '''
        It initializes the class with the tokenizer and the dataset and splits the data into training, validation and test data
        Stratify by column is used to ensure that the distribution of the labels is the same in the training and validation data
        '''
        self.prompt_tokens = prompt_tokens
        self.tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
        self.dataset = load_dataset('imdb')
        self.random_seed = random_seed
        train_data = self.dataset['train']
        test_data = self.dataset['test']
        train_val_split = train_data.train_test_split(test_size=0.1, seed=self.random_seed, stratify_by_column='label')
        self.train_data = train_val_split['train']
        self.val_data = train_val_split['test']
        self.test_data = test_data
        if clean_text:
            self.train_data = self.train_data.map(lambda example: {'text': self.clean_text(example['text'])})
            self.val_data = self.val_data.map(lambda example: {'text': self.clean_text(example['text'])})
            self.test_data = self.test_data.map(lambda example: {'text': self.clean_text(example['text'])})
    
    def tokenize_function(self,examples):
        '''
        Tokenize the text column of the input examples
        Cuts the text if it is longer than the maximum length of the model
        The responsibilty of the padding is left to the dataloader
        '''
        return self.tokenizer(examples['text'], truncation=True,max_length=512-self.prompt_tokens)
    
    def get_tokenized_datasets(self):
        '''
        Tokenize the training, validation and test data
        '''
        tokenized_train_data = self.train_data.map(self.tokenize_function, batched=True)
        tokenized_val_data = self.val_data.map(self.tokenize_function, batched=True)
        tokenized_test_data = self.test_data.map(self.tokenize_function, batched=True)
        
        tokenized_train_data = tokenized_train_data.remove_columns(['text'])
        tokenized_val_data = tokenized_val_data.remove_columns(['text'])
        tokenized_test_data = tokenized_test_data.remove_columns(['text'])
        
        return tokenized_train_data, tokenized_val_data, tokenized_test_data
    
    
    def clean_text(self,text):
        '''
        It removes the html tags from the text and replaces multiple spaces with a single space
        '''
        import re
        from bs4 import BeautifulSoup
        # Remove HTML tags
        text = BeautifulSoup(text, 'html.parser').get_text()
        #Remove multiple spaces,newlines and tabs with a single space
        text = re.sub(r'\n\s*\n+', '\n', text) # Remove multiple newlines with a single newline
        text = re.sub(r'(?<=\S)[ ]{2,}', ' ', text)  # Ensures only consecutive spaces collapse

        return text.strip()
    
    @classmethod
    def compute_metric_accuracy(cls,eval_pred):
        '''
        It computes the accuracy of the model
        '''
        accuracy = evaluate.load('accuracy')
        logits, labels = eval_pred
        predictions = logits.argmax(axis=-1)
        return accuracy.compute(predictions=predictions,references=labels)
    
    @classmethod
    def write_time(cls,start_time,end_time,batch_size,epochs):
        '''
        It writes the time taken to train the model
        '''
        with open('time.txt','a+') as f:
            f.write(f'Batch size {batch_size}:epochs={epochs}:{int(end_time-start_time)} seconds\n')

In [ ]:
import sys
import torch
from time import time
from transformers import set_seed

In [ ]:
set_seed(42)
torch.manual_seed(42)
batch_size = 32
epochs = 3
lr=5e-5
shuffle = True
clean_text = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
selection_criterion = 'eval_loss' # Choose between 'accuracy' and 'eval_loss'
prompt_tokens = 0

In [ ]:
base_utils_obj = BaseUtilsImdb(clean_text=clean_text,prompt_tokens=prompt_tokens)
tokenized_train, tokenized_val, tokenized_test = base_utils_obj.get_tokenized_datasets()
results_path = './results/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = 'distilbert-base-uncased'
    
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
rank = 16
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=rank,               # LoRA rank: 16 works better for text classification
    lora_alpha=2*rank,       # Scaling factor: standard is alpha ≈ 2 × r
    lora_dropout=0.05,   # Lower dropout for classification tasks (IMDB is not extremely noisy)
    target_modules=["q_lin", "v_lin" , "k_lin"],  # Focus only on query and value projections in DistilBERT
)

peft_config.modules_to_save = None

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()


In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=256,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=50,
    eval_strategy='steps',
    save_steps=50,
    eval_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model=selection_criterion,
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}, selection_criterion: {}'.format(batch_size, epochs, lr, selection_criterion),
    lr_scheduler_type='constant',
    warmup_steps=0,
    fp16=True,
    label_names=["labels"]
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=base_utils_obj.tokenizer,
    compute_metrics=base_utils_obj.compute_metric_accuracy,
)

In [ ]:
start_time = time()

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}_selection_criterion_{selection_criterion}".format(batch_size=batch_size, epochs=epochs, lr=lr, selection_criterion=selection_criterion)
trainer.model.save_pretrained(best_model_path)

In [ ]:
base_utils_obj.write_time(start_time,time(),batch_size,epochs)

In [ ]:
predictions = trainer.predict(tokenized_test)
accuracy = predictions[-1]['test_accuracy']

In [ ]:
with open('./test_acc.txt','a+') as f:
    f.write('Batch size: {}, Epochs: {}, LR: {}, selection_criterion: {}, Accuracy: {}\n'.format(batch_size, epochs, lr, selection_criterion, accuracy))